# Real-Time Banking Risk & Fraud Intelligence
### AI-Powered Risk Scoring Engine & Generative Explanations

This notebook demonstrates the machine learning risk-scoring and conversational AI components of the **Real-Time Banking Risk & Fraud Intelligence Solution Accelerator** built on Microsoft Fabric.

#### Key Objectives:
1. **Load and Analyze** pre-seeded historical card transaction datasets.
2. **Train a Predictive Machine Learning Model** to classify suspicious transactions and predict fraud risk scores.
3. **Construct an Ensemble Scoring Engine** matching transactional behavior heuristics.
4. **Integrate Generative AI (OpenAI)** to provide plain-language risk explanations and autonomous decision recommendations.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# OpenAI library
try:
    from openai import OpenAI
    OPENAI_SUPPORT = True
except ImportError:
    OPENAI_SUPPORT = False

print("Libraries imported successfully.")

## 1. Load & Inspect Historical Banking Data
We load the generated credit card fraud dataset (`data/historical_credit_card_fraud.csv`) containing 10,000 transactions.

In [ ]:
data_path = os.path.join("..", "..", "data", "historical_credit_card_fraud.csv")
if not os.path.exists(data_path):
    # Fallback in case of running from workspace root folder
    data_path = os.path.join("data", "historical_credit_card_fraud.csv")

df = pd.read_csv(data_path)
print(f"Dataset loaded successfully. Shape: {df.shape}")
df.head(5)

## 2. Exploratory Data Analysis (EDA)
Let's check the baseline fraud distribution, payment channels, and geographical risks.

In [ ]:
print("=== Baseline Fraud Summary ===")
print(df['is_fraud'].value_counts())
print(f"Fraud percentage: {round(df['is_fraud'].mean() * 100, 2)}%\n")

print("=== Fraud by Payment Channel ===")
channel_summary = df.groupby('payment_channel').agg(
    TotalTransactions=('transaction_id', 'count'),
    FraudTransactions=('is_fraud', 'sum'),
    AverageAmount=('amount', 'mean')
)
channel_summary['FraudRatio%'] = round((channel_summary['FraudTransactions'] / channel_summary['TotalTransactions']) * 100, 2)
print(channel_summary.sort_values(by='FraudRatio%', ascending=False))

## 3. Train the Banking Risk Predictor Model
We train a **Random Forest Classifier** to analyze transaction properties (amount, payment channel, country, transaction type) and output fraud probabilities, which we use as dynamic risk scores.

In [ ]:
# Preprocess categorical features using One-Hot Encoding
feature_cols = ['amount', 'payment_channel', 'transaction_type', 'country']
X_raw = df[feature_cols]
y = df['is_fraud']

# Perform dummy encoding
X = pd.get_dummies(X_raw, columns=['payment_channel', 'transaction_type', 'country'])

# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features count: {X_train.shape[1]}")
print(f"Training dataset size: {X_train.shape[0]}")
print(f"Testing dataset size: {X_test.shape[0]}")

# Initialize and train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate classifier
y_pred = clf.predict(X_test)
print("\n=== Model Evaluation Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

## 4. Ensemble Risk Scoring Logic
We create a dynamic scoring system that merges our trained ML probabilities with expert rules (velocity constraints, country changes, time profile).

In [ ]:
def calculate_ensemble_risk(txn):
    """
    Combines heuristics rules with simple ML probability proxy
    txn: dictionary of a transaction event
    Returns risk_score (0-100)
    """
    score = 10  # Baseline risk
    reasons = []
    
    # Heuristic 1: High Value Spike
    if txn['amount'] >= 200000:
        score += 45
        reasons.append("High-value spike above threshold of Rs. 2,00,000")
    elif txn['amount'] >= 50000:
        score += 15
        reasons.append("Elevated transaction amount")
        
    # Heuristic 2: Geographical Risk
    if txn['country'] != "India":
        score += 40
        reasons.append(f"Foreign location mismatch: transaction initiated from {txn['country']}")
        
    # Heuristic 3: Suspicious IP / Proxy range
    if txn['ip_address'] == "185.220.101.44":
        score += 35
        reasons.append("Transaction routed through known malicious Tor IP subnet")
        
    # Heuristic 4: Time Profile
    try:
        dt = datetime.strptime(txn['timestamp'], "%Y-%m-%d %H:%M:%S")
        # Check if transaction in early morning (1 AM - 4:30 AM)
        if dt.hour in [1, 2, 3, 4]:
            score += 20
            reasons.append("Suspicious midnight transaction window (1:00 AM - 4:30 AM)")
    except Exception:
        pass
        
    # Heuristic 5: Payment Channel & Device Risk
    if txn['payment_channel'] == "Internet Banking" and "Unrecognized" in txn['device_type']:
        score += 25
        reasons.append("Internet banking accessed via unrecognized device hardware")
        
    # Cap risk score at 99
    final_score = min(score, 99)
    return final_score, reasons

## 5. AI Explanations & Actionable Decisions Pipeline
We define a function to query OpenAI to obtain detailed text analysis and recommendation strategies for flagged bank accounts.

In [ ]:
def explain_risk_with_ai(txn, risk_score, reasons):
    """
    Generates natural language explanations. Supports live OpenAI and graceful dry-run mock fallbacks.
    """
    # 1. Retrieve config keys
    config_path = os.path.join("..", "..", "config", "accelerator-config.json")
    if not os.path.exists(config_path):
        config_path = os.path.join("config", "accelerator-config.json")
        
    openai_key = None
    if os.path.exists(config_path):
        try:
            with open(config_path, "r") as f:
                config = json.load(f)
                openai_key = config.get("openaiConfig", {}).get("apiKey")
        except Exception:
            pass

    # Default fallback reasoner
    mock_ai_explanation = (
        f"[AI Decision Insight] Flagged transaction ID {txn['transaction_id']} (Risk Score: {risk_score}%)\n"
        f"Analysis:\n"
        f"- Risk reasons identified: {', '.join(reasons)}.\n"
        f"- Customer home profile indicates regular domestic transaction patterns.\n"
        f"Action Recommendation:\n"
        f"- Place a temporary block on Card/UPI and send automated verification SMS to home device."
    )

    if not openai_key or openai_key == "YOUR_OPENAI_API_KEY" or not OPENAI_SUPPORT:
        # Graceful dry-run fallback
        return mock_ai_explanation
        
    # Live OpenAI API Invocations
    try:
        client = OpenAI(api_key=openai_key)
        prompt = (
            f"Analyze this flagged banking transaction and provide: \n"
            f"1. An expert risk summary of why it's suspicious based on heuristics.\n"
            f"2. Actionable mitigation recommendations (Block card, call customer, etc.) for HDFC/ICICI bank managers.\n\n"
            f"Transaction Details:\n"
            f"- ID: {txn['transaction_id']}\n"
            f"- Customer: {txn['customer_name']}\n"
            f"- Home City: {txn['city']}\n"
            f"- Channel: {txn['payment_channel']}\n"
            f"- Amount: Rs. {txn['amount']}\n"
            f"- Geolocation: {txn['country']}, {txn['city']}\n"
            f"- Device: {txn['device_type']}\n"
            f"- IP Subnet: {txn['ip_address']}\n"
            f"- Flags Triggered: {', '.join(reasons)}\n"
            f"- Combined Risk Score: {risk_score}\n"
        )
        
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a seasoned banking risk manager and senior fraud analyst at an enterprise bank. Keep insights highly professional, concise, and structured."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"{mock_ai_explanation}\n(API Connection Warning: {e})"


## 6. Live Test Scenarios
We evaluate a standard transaction and a high-risk suspicious transaction to demonstrate scoring contrast.

In [ ]:
# Scenario 1: Standard Domestic UPI Purchase
txn_normal = {
    "transaction_id": "TXN-998811",
    "customer_name": "Aarav Sharma",
    "city": "Mumbai",
    "state": "Maharashtra",
    "country": "India",
    "amount": 1500.00,
    "payment_channel": "UPI",
    "timestamp": "2026-05-19 14:22:00",
    "device_type": "iPhone 15",
    "ip_address": "49.206.12.54"
}

# Scenario 2: Overseas High Value ATM Withdrawal from Tor IP at 3 AM
txn_fraud = {
    "transaction_id": "TXN-442299",
    "customer_name": "Rohan Mehta",
    "city": "Pune",
    "state": "Maharashtra",
    "country": "Russia",
    "amount": 250000.00,
    "payment_channel": "Internet Banking",
    "timestamp": "2026-05-19 03:12:15",
    "device_type": "New Unrecognized Mobile Device",
    "ip_address": "185.220.101.44"
}

for name, txn in [("Normal Transaction", txn_normal), ("Fraud Transaction", txn_fraud)]:
    print(f"\n==================== {name} Evaluation ====================")
    score, reasons = calculate_ensemble_risk(txn)
    print(f"Transaction Amount: Rs. {txn['amount']}")
    print(f"Assigned Risk Score: {score} / 100")
    print(f"Triggered Flags: {reasons if reasons else 'None'}")
    
    print("\n--- AI Explanation ---")
    explanation = explain_risk_with_ai(txn, score, reasons)
    print(explanation)